# Section 8: Validate Colocalization Results in Other Contexts

This notebook documents whether the colocalized results for the target gene can be validated in other contexts.

The steps are:
1. Load individual QTL data. Currently, we only have extracted data for ROSMAP, but you can load exported summary statistics (sumstats) data for other contexts, which is also supported by ColocBoost.
2. Load the interested finemapping contexts, which is from Section 1.
3. Load QTL data or AD sumstats data for other contexts, and extract the corresponding LD matrix. **Note:** The LD should be in matrix format, and the sumstats should be in data.frame format.
4. Run ColocBoost with each AD sumstats data in a for loop.

To apply this to other genes, simply use `Ctrl+F` to replace `BIN1` with the gene of interest, or change the gene name in the block below.

In [1]:
# define function 
library(tidyverse)
library(data.table)
library(pecotmr) # micromamba install r-pecotmr -c dnachun


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose




In [2]:
source('../../codes/utilis.R')
gene_name = "BIN1"
tar_gene_info <- get_gene_info(gene_name = gene_name)
gene_info <- tar_gene_info$gene_info
gene_id <- gene_info$region_id
target_sums_ids <- tar_gene_info$target_sums_ids
gene_region <- tar_gene_info$gene_region

## xQTL data results 

In [17]:
# get extracted residule data
Extracted_eQTL <- list.files("/data/colocalization/QTL_data/eQTL/", pattern = ".rds", full.names = TRUE)
Extracted_sQTL <- list.files("/data/colocalization/QTL_data/sQTL/", pattern = ".rds", full.names = TRUE)
all_pheno <- c("Mic","Ast","Oli","OPC","Exc","Inh", "DLPFC", "AC", "PCC", "Monocyte",
                   "pQTL", "AC_unproductive", "DLPFC_unproductive", "PCC_unproductive")
# - QTL data
pos.extract.eQTL <- grep(gene_id, Extracted_eQTL)
pos.extract.sQTL <- grep(gene_id, Extracted_sQTL)

if (length(pos.extract.eQTL)!=0){
    ifempty.eQTL <- file.size(Extracted_eQTL[pos.extract.eQTL]) == 0
    if (!ifempty.eQTL){
        QTL <- readRDS(Extracted_eQTL[pos.extract.eQTL])
        pos <- na.omit(match(all_pheno, names(QTL[[1]]$residual_Y)))
        phenotypes.eQTL <- names(QTL[[1]]$residual_Y)[pos]
        X <- QTL[[1]]$residual_X[pos]
        Y.eQTL <- QTL[[1]]$residual_Y[pos]
        # - remove variants with maf < 0.005
        maf <- QTL[[1]]$maf[pos]
        X_rm <- lapply(1:length(X), function(ii){
            x.tmp <- X[[ii]]
            maf.tmp <- maf[[ii]]
            pp <- which(maf.tmp < 0.005)
            snp.name <- colnames(x.tmp)
            if (length(pp)!=0){
                x_rm <- x.tmp[, -pp, drop = FALSE]
                colnames(x_rm) <- snp.name[-pp]
                rownames(x_rm) <- rownames(x.tmp)
                return(x_rm)
            } else {
                return(x.tmp)
            }
        })
    } else {
        X_rm <- Y.eQTL <- phenotypes.eQTL <- NULL
    }
} else {
    X_rm <- Y.eQTL <- phenotypes.eQTL <- NULL
}
if (length(pos.extract.sQTL)!=0){
    ifempty.sQTL <- file.size(Extracted_sQTL[pos.extract.sQTL]) == 0
    if (!ifempty.sQTL){
        sQTL <- readRDS(Extracted_sQTL[pos.extract.sQTL])
        pos <- na.omit(match(all_pheno, names(sQTL[[1]]$residual_Y)))
        if (length(pos) != 0){
            phenotypes.xQTL <- names(sQTL[[1]]$residual_Y)[pos]
            X_sQTL <- sQTL[[1]]$residual_X[pos]
            Y_sQTL <- sQTL[[1]]$residual_Y[pos]
            # - remove variants with maf < 0.005
            maf <- sQTL[[1]]$maf[pos]
            X_rm_sQTL <- lapply(1:length(X_sQTL), function(ii){
                x.tmp <- X_sQTL[[ii]]
                maf.tmp <- maf[[ii]]
                pp <- which(maf.tmp < 0.005)
                snp.name <- colnames(x.tmp)
                if (length(pp)!=0){
                    x_rm <- x.tmp[, -pp, drop = FALSE]
                    colnames(x_rm) <- snp.name[-pp]
                    rownames(x_rm) <- rownames(x.tmp)
                    return(x_rm)
                } else {
                    return(x.tmp)
                }
            })
            # - extract pQTL
            if_pQTL <- "pQTL" %in% phenotypes.xQTL
            if (if_pQTL){
                pp <- which(phenotypes.xQTL == "pQTL")
                X_rm_pQTL <- X_rm_sQTL[pp]
                Y_pQTL <- Y_sQTL[pp]
                X_rm_sQTL <- X_rm_sQTL[-pp]
                Y_sQTL <- Y_sQTL[-pp]
                phenotypes.pQTL <- "pQTL"
            } else {
                X_rm_pQTL <- Y_pQTL <- phenotypes.pQTL <- NULL
            }
            # - extract sQTL select top event based on the smallest p-value
            if (length(Y_sQTL)!=0){
                region <- do.call(rbind, strsplit(names(Y_sQTL), "_"))[,1]
                p.re <- match(region, c("AC", "DLPFC", "PCC"))
                X_sQTL_dup <- vector("list", length = 6)
                X_sQTL_dup[sort(c(p.re*2-1, 2*p.re))] <- X_rm_sQTL[rep(1:length(X_rm_sQTL), each=2)]

                sQTL.top <- read.csv("/data/colocalization/QTL_data/top_new.csv")
                p.top <- which(sQTL.top$X == gene_id)
                top_sign <- sQTL.top[p.top,-1]
                if (sum(is.na(top_sign))!=6){
                    top_events <- paste0(c("AC:", "AC:", "DLPFC:", "DLPFC:", "PCC:", "PCC:"), top_sign)
                    events_name <- lapply(1:length(Y_sQTL), function(i.y){
                        region <- strsplit(names(Y_sQTL)[i.y], "_")[[1]][1]
                        if (ncol(Y_sQTL[[i.y]])!=1){
                            paste0(region, ":", colnames(Y_sQTL[[i.y]]))
                        } else {
                            tmp <- na.omit(unlist(top_sign[grep(region, names(top_sign))]))
                            paste0(region, ":", tmp)
                        }
                    })
                    top_pos <- lapply(events_name, function(tmp) na.omit(match(top_events, tmp)))
                    Y_sQTL_comb <- lapply(1:length(top_pos), function(i.p){
                        pos.tmp <- top_pos[[i.p]]
                        if (length(pos.tmp)!=0){
                            y <- c()
                            for (ii in pos.tmp){
                                y.tmp <- Y_sQTL[[i.p]][, ii, drop = FALSE]
                                colnames(y.tmp) <- events_name[[i.p]][ii]
                                y <- c(y, list(y.tmp))
                            }
                            return(y)
                        }
                    })
                    Y_sQTL_comb <- Reduce("c", Y_sQTL_comb)
                    phenotypes.sQTL <- names(top_sign)[!is.na(top_sign)]
                    X_sQTL_comb <- X_sQTL_dup[!is.na(top_sign)]
                } else {
                    X_sQTL_comb <- Y_sQTL_comb <- phenotypes.sQTL <- NULL
                }
            } else {
                X_sQTL_comb <- Y_sQTL_comb <- phenotypes.sQTL <- NULL
            }

        } else {
            X_rm_pQTL <- Y_pQTL <- phenotypes.pQTL <- NULL
            X_sQTL_comb <- Y_sQTL_comb <- phenotypes.sQTL <- NULL
        }
    } else {
        X_rm_pQTL <- Y_pQTL <- phenotypes.pQTL <- NULL
        X_sQTL_comb <- Y_sQTL_comb <- phenotypes.sQTL <- NULL
    }
} else {
    X_rm_pQTL <- Y_pQTL <- phenotypes.pQTL <- NULL
    X_sQTL_comb <- Y_sQTL_comb <- phenotypes.sQTL <- NULL
}
X <- c(X_rm, X_rm_pQTL, X_sQTL_comb)
Y <- c(Y.eQTL, Y_pQTL, Y_sQTL_comb)
phenotypes <- c(phenotypes.eQTL, phenotypes.pQTL, phenotypes.sQTL)

In [ ]:
# save(X, Y, file = "data_for_cb_BIN1.Rdata")
# load("data_for_cb_BIN1.Rdata")

## LD

In [4]:
LD_all <- load_LD_matrix(LD_meta_file_path = '/data/ADSP_R4_EUR_LD/ld_meta_file.tsv', region = gene_region)
LD_all_df <- LD_all$combined_LD_matrix
colnames(LD_all_df) <- rownames(LD_all_df) <- LD_all$combined_LD_variants %>% paste0('chr',.)

In [6]:
# save(X, Y, LD_all_df, file = "data_for_cb_BIN1.Rdata")

## MiGA data

In [30]:
gene_sum <- readRDS(paste0('/data/analysis_result/finemapping_twas/susie_twas_export/Fungen_xQTL.',gene_id,'.cis_results_db.export_sumstats.rds'))

In [32]:
miga_contexts <- gene_sum[[1]] %>% names %>% .[str_detect(., 'MiGA')]

In [51]:
miga_contexts

[1] "MiGA_GFM_eQTL" "MiGA_GTS_eQTL" "MiGA_SVZ_eQTL" "MiGA_THA_eQTL"

In [21]:
LD_all_df <- LD$MiGA 

In [40]:
miga_LD <- miga_sum <- list()
for(miga_context in miga_contexts){
  dat <- gene_sum[[1]][[miga_context]]
    
    miga_sum_tmp <- data.frame(n_sample_median = 74, ##FIXME if not miga data
              variant = dat$variant_names,
              z = dat[['sumstats']]$betahat/dat[['sumstats']]$sebetahat) %>% 
            mutate(pos = str_split(variant, ":", simplify = T) %>% .[,2]) %>%
      filter(pos >= gene_info$start, pos <= gene_info$end) %>% 
      select(z = z, n = n_sample_median, variant) 
    
    vars <- intersect(colnames(LD_all_df), miga_sum_tmp$variant)
 
    miga_LD <- c(miga_LD, list(LD_all_df[vars, vars] %>% as.matrix))
    
    miga_sum <- c(miga_sum, list(miga_sum_tmp %>% 
      filter(variant %in% vars) %>%
      mutate(order = match(variant, vars)) %>%
      arrange(order) %>%
      select(-order) %>% 
      as.data.frame))
  
}



In [ ]:
# # save(miga_sum, miga_LD, X, Y, LD_all_df, file = "data_for_cb_BIN1.Rdata")

# load("/data/interactive_analysis/rf2872/FunGen_xQTL/Sep/data_for_cb_BIN1_new.Rdata")

## AD data

In [4]:
cohorts <- c('AD_Bellenguez_2022'
             # ,'AD_Bellenguez_EADB_2022', 'AD_Bellenguez_EADI_2022', 'AD_Jansen_2021', 
             # 'AD_Kunkle_Stage1_2019', 'AD_Wightman_Excluding23andMe_2021',
             # 'AD_Wightman_ExcludingUKBand23andME_2021', 'AD_Wightman_Full_2021'
            )

<!-- ### load sumstats

cohort = cohorts[3]
# find gwas sumstats file path 
gwas_path = '/data/interactive_analysis/rf2872/resource/GWAS_Apr9'
file = paste0(gwas_path, '/', cohort,'_RSS_QC_RAISS_imputed.tsv.gz')
# load gwas results with specific region
gwas <- pecotmr:::tabix_region(file = file, 
             region =  gene_region %>% gsub('chr','',.))


# tabix would lost the header of the file , add it manually 
header <- readLines(pipe(paste0("zcat ", file, " | head -n 1")))
header <- unlist(strsplit(header, "\t"))  

colnames(gwas) <- header

# add a column n_sample if that is seperated in Bellenguez data
if(cohort == 'AD_Kunkle_Stage1_2019'){
    gwas <- gwas %>% mutate(n_sample = 94437)
} else {
    if(!("n_sample" %in% colnames(gwas))) gwas <- gwas %>% mutate(n_sample = n_case + n_control)
}

gwas <- gwas %>% 
  mutate(n_sample_median = median(.data[['n_sample']], na.rm = T), 
        variant = paste0('chr', variant_id)) %>%
  filter(pos >= gene_info$start, pos <= gene_info$end) %>% 
  select(z = z, n = n_sample_median, variant)

# - all AD GWAS
ADGWAS_sumstat <- list(gwas %>% as.data.frame)
names(ADGWAS_sumstat) <- cohort

### load LD

LD_all <- load_LD_matrix(LD_meta_file_path = '/data/ADSP_R4_EUR_LD/ld_meta_file.tsv', region = gene_region)

LD_all_df <- LD_all$combined_LD_matrix
colnames(LD_all_df) <- rownames(LD_all_df) <- LD_all$combined_LD_variants %>% paste0('chr',.)

gwas_LD <- LD_all_df[gwas$variant, gwas$variant]

# all LD
LD <- list(gwas_LD %>% as.matrix)
names(LD) <- cohort









LD %>% str

ADGWAS_sumstat %>% str

## run colocboost

class(LD[[1]])

for(file in list.files("/data/colocalization/colocboost/R/", pattern = ".R", full.names = T)){
  source(file)
}

res_z2z <- colocboost(X = X, Y = Y, sumstat = ADGWAS_sumstat, LD = LD, same_idx_cut = 0.85)

results <- list("phenotypes" = c(phenotypes, names(ADGWAS_sumstat)),
                            "cb_z2z_noLD" = res_z2z)

get_summary_table(results)

dir.create('./BIN1/')
saveRDS(results, paste0("./BIN1/BIN1_", names(ADGWAS_sumstat), "_idx085.rds")) -->

## for loop with add MiGA data

In [ ]:
for(file in list.files("/data/colocalization/colocboost/R", pattern = ".R", full.names = T)){
  source(file)
}
library(tidyverse)
 library(pecotmr)
## QTL data is ready

## AD data

### load sumstats

for(cohort in cohorts[1]){
    message(cohort)
    # find gwas sumstats file path 
    gwas_path = '/data/GWAS/AD_GWAS_RSS_QC_RAISS_imputed_concatenate_sumstat'
    file = paste0(gwas_path, '/', cohort,'_RSS_QC_RAISS_imputed.tsv.gz')
    # load gwas results with specific region
    # FIXME: 'AD_Bellenguez_EADB_2022', 'AD_Bellenguez_EADI_2022' do not have tbi in concatenat files
    if(cohort %in% c('AD_Bellenguez_EADB_2022', 'AD_Bellenguez_EADI_2022')){
        # prepare sumstats
        gwas <- gwas_tmp <- data.frame()
        for(block in target_sums_ids) {
            try({
                gwas_tmp <- fread(paste0('/data/GWAS/ADGWAS_sumstats/', block, '.RSS_QC_RAISS_imputed.', cohort, '.sumstats.tsv.gz'))
                gwas <- rbind(gwas, gwas_tmp, fill = TRUE)
            })
        }  
    }else {
        gwas <- pecotmr:::tabix_region(file = file, 
                     region =  gene_region %>% gsub('chr','',.))
        # tabix would lost the header of the file , add it manually 
        header <- readLines(pipe(paste0("zcat ", file, " | head -n 1")))
        header <- unlist(strsplit(header, "\t"))  
        colnames(gwas) <- header
    }
    
    # add a column n_sample if that is seperated in Bellenguez data
    if(cohort == 'AD_Kunkle_Stage1_2019'){
        gwas <- gwas %>% mutate(n_sample = 94437)
    } else {
        if(!("n_sample" %in% colnames(gwas))) gwas <- gwas %>% mutate(n_sample = n_case + n_control)
    }    
    median_nsample = median(gwas[['n_sample']], na.rm = T)
    gwas <- gwas %>% 
      mutate(n_sample_median = median_nsample, 
            variant = paste0('chr', variant_id)) %>%
      filter(pos >= gene_info$start, pos <= gene_info$end) %>% 
      select(z = z, n = n_sample_median, variant) %>% mutate()
    # - all AD GWAS
    ADGWAS_sumstat <- c(miga_sum , list(gwas %>% as.data.frame))
    names(ADGWAS_sumstat) <- c(miga_contexts,cohort)
    
    ### load LD
    gwas_LD <- LD_all_df[gwas$variant, gwas$variant]
    # all LD
    LD <- c(miga_LD , list(gwas_LD %>% as.matrix))
    names(LD) <- c(miga_contexts, cohort)
        
        
    ## run colocboost
    message('runnning colocboost')
    res_z2z <- colocboost(X = X, Y = Y, sumstat = ADGWAS_sumstat, LD = LD, same_idx_cut = 0.85)
    
    results <- list("phenotypes" = c(phenotypes, names(ADGWAS_sumstat)),
                                "cb_z2z_noLD" = res_z2z)
    
    dir.create(gene_id)
    saveRDS(results, paste0(gene_id, "/AddMiga_",gene_id, "_", cohort, "_idx085.rds"))
}

AD_Bellenguez_2022



In [15]:
cb_res = results

In [ ]:
source('../../codes/cb_plot.R')
cb <- plot_cb(cb_res = cb_res, cex.pheno = 1.5, x.phen = -0.2)

In [ ]:
cb2 <- plot_cb(cb_res, cex.pheno = 1.5,fake_contexts = c('MiGA_rss'), fake_targets = c('AD_Bellenguez_2022'),  x.phen = -0.2)

In [49]:
# save(ADGWAS_sumstat, LD, X, Y, phenotypes, file = "data_for_cb_BIN1_new.Rdata")